In [1]:
from google.colab import files
files.upload()          # choose StegoShield-main.zip
!unzip -q StegoShield-main.zip
%cd StegoShield-main

Saving StegoShield-main.zip to StegoShield-main.zip
/content/StegoShield-main


In [4]:
p = "stego/extract.py"
s = open(p).read()
start = s.index("    # 3) decrypt")
end = s.index('        return plaintext.decode("utf-8")')
end = s.rindex("try:", start, end)
end = s.rindex("\n", start, end) + 1
good = '''    # 3) decrypt (raises ValueError if the password is wrong or data changed)
    try:
        plaintext = crypto.decrypt(payload, password)
    except ValueError:
        if mode == "random":
            raise ValueError("Hidden data was found, but it is damaged. "
                             "The image was edited after embedding.")
        raise ValueError("Hidden data was found, but it could not be decrypted. "
                         "The password is wrong, or the image was edited.")
'''
s = s[:start] + good + s[end:]
s = s.replace('raise ValueError("No hidden data found, or wrong password")',
              'raise ValueError("No hidden message found. Either this image has nothing hidden in it "\n'
              '                         "(for example, the original cover image), or the password is wrong.")')
open(p, "w").write(s)
print("extract.py fixed")

extract.py fixed


In [5]:
!pip install -q -r requirements.txt

In [6]:
!pytest -q
!python main.py embed data/cover_images/astronaut.png data/stego_images/out.png "Hello" 1234 --mode random
!python main.py extract data/stego_images/out.png 1234 --mode random
!python main.py detect data/stego_images/out.png

..........                                                               [100%]
10 passed in 5.75s
{'capacity_bits': 786432, 'used_bits': 424, 'mode': 'random'}
Hello
{'likely_hidden': False, 'confidence': 1.0, 'chi_square_p': 0.0, 'lsb_ratio': 0.4485270182291667, 'pair_diff': 1.0}


In [ ]:
!pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/StegoShield-main/StegoShield-main/StegoShield-main
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 10 items                                                             

tests/test_crypto.py::test_roundtrip PASSED                              [ 10%]
tests/test_crypto.py::test_wrong_password_rejected PASSED                [ 20%]
tests/test_crypto.py::test_tampered_data_rejected PASSED                 [ 30%]
tests/test_detector.py::test_detect_output_format PASSED                 [ 40%]
tests/test_detector.py::test_detects_heavy_embedding PASSED              [ 50%]
tests/test_detector.py::test_clean_image_not_flagged PASSED              [ 60%]
tests/test_embed_extract.py::test_embed_then_extract[sequential] PASSED  [ 70%]
tests/test_embed_extract.

In [7]:
# Embed
!python main.py embed data/cover_images/a.png data/stego_images/stego_image.png "Happy Birthday" 87654321 --mode random

# Extract (correct password)
!python main.py extract data/stego_images/stego_image.png 87654321 --mode random

# Extract (wrong password, should give an error)
!python main.py extract data/stego_images/stego_image.png 11111111 --mode random

# Detect
!python main.py detect data/stego_images/stego_image.png

{'capacity_bits': 196608, 'used_bits': 496, 'mode': 'random'}
Happy Birthday
Error: No hidden message found. Either this image has nothing hidden in it (for example, the original cover image), or the password is wrong.
{'likely_hidden': False, 'confidence': 1.0, 'chi_square_p': 5.772571175143157e-186, 'lsb_ratio': 0.4986572265625, 'pair_diff': 1.0}


In [8]:
!pkill -f streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!streamlit run gui/app.py --server.enableCORS false --server.enableXsrfProtection false &>/content/log.txt &
!./cloudflared tunnel --url http://localhost:8501

2026-11-28T05:42:36Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-28T05:42:36Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-28T05:42:41Z INF +--------------------------------------------------------------------------------------------+
2026-11-28T05:42:41Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-28T05:42:41Z INF |  https://inspired-lime-informed-estates.trycloudflare.